In [1]:
import pandas
from sklearn import linear_model

df = pandas.read_csv("cars.csv")

X = df[['Weight', 'Volume']]
y = df['CO2']

regr = linear_model.LinearRegression()
regr.fit(X, y)

#predict the CO2 emission of a car where the weight is 2300kg, and the volume is 1300cm3:
predictedCO2 = regr.predict([[2300, 1300]])

print(predictedCO2)

[107.2087328]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


# Coefficient

The coefficient is a factor that describes the relationship with an unknown variable.
In this case, we can ask for the coefficient value of weight against CO2, and for volume against CO2. The answer(s) we get tells us what would happen if we increase, or decrease, one of the independent values.

In [2]:
print(regr.coef_)

[0.00755095 0.00780526]


The result array represents the coefficient values of weight and volume.

Weight: 0.00755095
Volume: 0.00780526

These values tell us that if the weight increase by 1kg, the CO2 emission increases by 0.00755095g.

And if the engine size (Volume) increases by 1 cm3, the CO2 emission increases by 0.00780526 g.

I think that is a fair guess, but let test it!

We have already predicted that if a car with a 1300cm3 engine weighs 2300kg, the CO2 emission will be approximately 107g.

What if we increase the weight with 1000kg (from 2300 to 3300) what will be the CO2 emission?

Ans: 107.2087328 + (1000 * 0.00755095) = 114.75968

In [3]:
predictedCO2 = regr.predict([[3300, 1300]])

print(predictedCO2)

[114.75968007]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


## Experiment: Changing the Data

The code below is identical to the code above; only the training data and the prediction inputs (the variables) are changed to see how the change in data impacts the regression coefficients and predictions. First, the model is refit on only the smaller/lighter cars in the data set, then on only the larger/heavier cars, and the same weight/volume prediction is compared across both.

In [4]:
import pandas
from sklearn import linear_model

df = pandas.read_csv("cars.csv")

#Use only the lighter cars in the data set (Weight < 1200)
df_light = df[df['Weight'] < 1200]

X = df_light[['Weight', 'Volume']]
y = df_light['CO2']

regr = linear_model.LinearRegression()
regr.fit(X, y)

#predict the CO2 emission of a car where the weight is 2300kg, and the volume is 1300cm3:
predictedCO2 = regr.predict([[2300, 1300]])

print(predictedCO2)
print(regr.coef_)


[89.49553615]
[-0.00740008  0.00831777]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


**Observation (lighter cars only):** Refitting on only the lighter cars changes the model in a striking way — the predicted CO2 for a 2300kg/1300cm3 car drops to about 89.5g (vs. 107.2g from the full model), and the weight coefficient actually turns *negative* (about -0.0074, vs. +0.00755 in the original). This happens because the training data no longer includes any cars near 2300kg, so the model is extrapolating far outside the range it was trained on — the negative coefficient doesn't reflect a real-world relationship, it's an artifact of fitting a straight line to a narrow slice of data and then projecting it well past that slice.

In [5]:
import pandas
from sklearn import linear_model

df = pandas.read_csv("cars.csv")

#Use only the heavier cars in the data set (Weight >= 1400)
df_heavy = df[df['Weight'] >= 1400]

X = df_heavy[['Weight', 'Volume']]
y = df_heavy['CO2']

regr = linear_model.LinearRegression()
regr.fit(X, y)

#predict the CO2 emission of a car where the weight is 2300kg, and the volume is 1300cm3:
predictedCO2 = regr.predict([[2300, 1300]])

print(predictedCO2)
print(regr.coef_)


[131.60838889]
[0.03754317 0.00579951]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


**Observation (heavier cars only):** Refitting on only the heavier cars again changes the coefficients and the prediction, this time in the other direction — the predicted CO2 rises to about 131.6g, and the weight coefficient roughly quintuples to about 0.0375 (vs. +0.00755 in the original), while the volume coefficient drops slightly to about 0.0058. Since this subset does include cars closer to the 2300kg used for prediction, this is a more reasonable interpolation than the lighter-cars model, but the smaller sample size still makes the coefficients noisier and more extreme than the coefficients from the full data set.

### Summary of Observations: How Data Changes Affect Multiple Linear Regression

Comparing the original model (fit on all 36 cars) with the lighter-cars-only and heavier-cars-only models:

| Model | Weight coef. | Volume coef. | Predicted CO2 @ 2300kg/1300cm3 |
|---|---|---|---|
| Full data set (36 cars) | 0.00755 | 0.00781 | 107.2 g |
| Lighter cars only (Weight < 1200) | -0.00740 | 0.00832 | 89.5 g |
| Heavier cars only (Weight >= 1400) | 0.03754 | 0.00580 | 131.6 g |

- **Coefficients shift — sometimes dramatically — when the training data changes.** The full model's weight coefficient (0.00755) flips sign entirely for the lighter-cars subset and roughly quintuples for the heavier-cars subset. This shows the relationship between weight/volume and CO2 isn't perfectly uniform across the whole range, and a smaller/narrower sample gives the model much less information to work with.
- **Predictions depend heavily on whether the input is inside the range of the training data.** Predicting CO2 for a 2300kg car using a model trained only on cars under 1200kg means the model is extrapolating well outside anything it has seen, which produced the nonsensical negative weight coefficient and an unreliable prediction. The heavier-cars model, which does include cars near 2300kg, produced a more plausible (if still noisier) prediction.
- **Smaller data sets produce less stable models.** With far fewer rows to fit (each subset is roughly half the original data set), both subset models are more sensitive to individual data points, so their coefficients and predictions swing more than the original model, which uses all 36 observations.

In short: changing which data points are used to train a multiple regression model changes both the coefficients and the resulting predictions — sometimes reversing the direction of a coefficient — and predictions are most trustworthy when the input values fall within the range of the data the model was actually trained on.